# 🛒 Walmart Weekly Sales Analysis
**Dataset:** 45 stores | 6,435 rows | Weekly sales from 2010–2012

---
### How to use this notebook
1. Place your `archive__1_.zip` (or `Walmart DataSet.csv`) in your **Downloads**, **Desktop**, **Documents**, or the **same folder as this notebook**
2. Run all cells top to bottom: **Kernel → Restart & Run All**
3. The final Excel report is saved in the same folder as this notebook

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import zipfile
import os
import warnings
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
print('✅ Imports done')

In [ ]:
# ── Cell 2: Load Data (auto-detects ZIP or CSV) ───────────────────────────────
def find_file(filename):
    search_dirs = [
        Path.cwd(),
        Path.home() / 'Downloads',
        Path.home() / 'Desktop',
        Path.home() / 'Documents',
    ]
    for d in search_dirs:
        p = d / filename
        if p.exists():
            return p
    return None

csv_path = find_file('Walmart DataSet.csv')
zip_path = find_file('archive__1_.zip') or find_file('archive.zip')

if csv_path:
    print(f'📄 Found CSV: {csv_path}')
    df = pd.read_csv(csv_path)
elif zip_path:
    print(f'📦 Found ZIP: {zip_path}')
    with zipfile.ZipFile(zip_path) as z:
        csv_name = next(n for n in z.namelist() if n.endswith('.csv'))
        with z.open(csv_name) as f:
            df = pd.read_csv(f)
else:
    # Fallback: manually paste path
    path = input("❌ File not found.\nPaste the full path to the ZIP or CSV file: ").strip().strip('"')
    path = Path(path)
    if path.suffix == '.zip':
        with zipfile.ZipFile(path) as z:
            csv_name = next(n for n in z.namelist() if n.endswith('.csv'))
            with z.open(csv_name) as f:
                df = pd.read_csv(f)
    else:
        df = pd.read_csv(path)

print(f'✅ Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ── Cell 3: Clean & Feature Engineering ──────────────────────────────────────
df['Date']         = pd.to_datetime(df['Date'], dayfirst=True)
df['Year']         = df['Date'].dt.year
df['Month']        = df['Date'].dt.month
df['Month_Name']   = df['Date'].dt.strftime('%b')
df['Week']         = df['Date'].dt.isocalendar().week.astype(int)
df['Holiday_Flag'] = df['Holiday_Flag'].map({0: 'Regular', 1: 'Holiday'})

print('📅 Date range:', df['Date'].min().date(), '→', df['Date'].max().date())
print('🏪 Stores    :', df['Store'].nunique())
print('❓ Nulls     :', df.isnull().sum().sum())
df.dtypes

In [ ]:
# ── Cell 4: Summary Statistics ────────────────────────────────────────────────
display(df[['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']].describe().round(2))

In [ ]:
# ── Cell 5: Store-level Summary ───────────────────────────────────────────────
store_summary = (
    df.groupby('Store')
    .agg(
        Total_Sales      = ('Weekly_Sales', 'sum'),
        Avg_Weekly_Sales = ('Weekly_Sales', 'mean'),
        Max_Weekly_Sales = ('Weekly_Sales', 'max'),
        Min_Weekly_Sales = ('Weekly_Sales', 'min'),
        Weeks_Count      = ('Weekly_Sales', 'count'),
        Holiday_Weeks    = ('Holiday_Flag', lambda x: (x == 'Holiday').sum()),
    )
    .reset_index()
    .sort_values('Total_Sales', ascending=False)
)
store_summary['Rank'] = range(1, len(store_summary) + 1)

print('🏆 Top 5 Stores by Total Sales:')
display(store_summary.head(5).style.format({'Total_Sales': '${:,.0f}', 'Avg_Weekly_Sales': '${:,.0f}'}))

In [ ]:
# ── Cell 6: Holiday vs Regular Sales ─────────────────────────────────────────
holiday_comp = (
    df.groupby('Holiday_Flag')
    .agg(
        Avg_Weekly_Sales = ('Weekly_Sales', 'mean'),
        Total_Sales      = ('Weekly_Sales', 'sum'),
        Count            = ('Weekly_Sales', 'count'),
    )
    .reset_index()
)
display(holiday_comp.style.format({'Avg_Weekly_Sales': '${:,.0f}', 'Total_Sales': '${:,.0f}'}))

In [ ]:
# ── Cell 7: Monthly Trend ─────────────────────────────────────────────────────
monthly = (
    df.groupby(['Year', 'Month', 'Month_Name'])
    .agg(Avg_Sales=('Weekly_Sales', 'mean'), Total_Sales=('Weekly_Sales', 'sum'))
    .reset_index()
    .sort_values(['Year', 'Month'])
)
display(monthly.head(10).style.format({'Avg_Sales': '${:,.0f}', 'Total_Sales': '${:,.0f}'}))

In [ ]:
# ── Cell 8: Correlation Matrix ────────────────────────────────────────────────
corr_cols    = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
corr_matrix  = df[corr_cols].corr().round(4)
display(corr_matrix.style.background_gradient(cmap='coolwarm', axis=None).format('{:.4f}'))

In [ ]:
# ── Cell 9: Yearly Performance ────────────────────────────────────────────────
yearly = (
    df.groupby('Year')
    .agg(
        Total_Sales      = ('Weekly_Sales', 'sum'),
        Avg_Weekly_Sales = ('Weekly_Sales', 'mean'),
        Store_Count      = ('Store', 'nunique'),
        Holiday_Weeks    = ('Holiday_Flag', lambda x: (x == 'Holiday').sum()),
    )
    .reset_index()
)
display(yearly.style.format({'Total_Sales': '${:,.0f}', 'Avg_Weekly_Sales': '${:,.0f}'}))

In [ ]:
# ── Cell 10: Charts ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Walmart Sales Analysis', fontsize=16, fontweight='bold')

# Chart 1 – Top 10 Stores
top10 = store_summary.head(10)
axes[0,0].barh(top10['Store'].astype(str), top10['Total_Sales'] / 1e6, color='#0D3B66')
axes[0,0].set_title('Top 10 Stores – Total Sales')
axes[0,0].set_xlabel('Total Sales ($M)')
axes[0,0].set_ylabel('Store')
axes[0,0].invert_yaxis()

# Chart 2 – Holiday vs Regular
hc = holiday_comp.set_index('Holiday_Flag')['Avg_Weekly_Sales']
axes[0,1].bar(hc.index, hc.values / 1e3, color=['#E8A838', '#0D3B66'])
axes[0,1].set_title('Avg Weekly Sales: Holiday vs Regular')
axes[0,1].set_ylabel('Avg Sales ($K)')
axes[0,1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}K'))

# Chart 3 – Monthly Average Sales
monthly_avg = df.groupby('Month')['Weekly_Sales'].mean()
axes[1,0].plot(monthly_avg.index, monthly_avg.values / 1e3, marker='o', color='#0D3B66', linewidth=2)
axes[1,0].set_title('Avg Weekly Sales by Month (All Years)')
axes[1,0].set_xlabel('Month')
axes[1,0].set_ylabel('Avg Sales ($K)')
axes[1,0].set_xticks(range(1, 13))
axes[1,0].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
axes[1,0].grid(True, alpha=0.3)

# Chart 4 – Sales vs Unemployment
axes[1,1].scatter(df['Unemployment'], df['Weekly_Sales'] / 1e3, alpha=0.3, color='#0D3B66', s=10)
axes[1,1].set_title('Weekly Sales vs Unemployment Rate')
axes[1,1].set_xlabel('Unemployment Rate (%)')
axes[1,1].set_ylabel('Weekly Sales ($K)')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('walmart_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Charts saved to walmart_charts.png')

In [ ]:
# ── Cell 11: Export to Excel ──────────────────────────────────────────────────
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

top5   = store_summary.head(5)[['Rank', 'Store', 'Total_Sales', 'Avg_Weekly_Sales']]
bottom5 = store_summary.tail(5)[['Rank', 'Store', 'Total_Sales', 'Avg_Weekly_Sales']]
output  = 'walmart_analysis_output.xlsx'

with pd.ExcelWriter(output, engine='openpyxl') as writer:
    df.to_excel(writer,           sheet_name='Raw Data',           index=False)
    store_summary.to_excel(writer, sheet_name='Store Summary',      index=False)
    holiday_comp.to_excel(writer,  sheet_name='Holiday vs Regular', index=False)
    monthly.to_excel(writer,       sheet_name='Monthly Trend',      index=False)
    corr_matrix.to_excel(writer,   sheet_name='Correlation Matrix')
    top5.to_excel(writer,          sheet_name='Top 5 Stores',       index=False)
    bottom5.to_excel(writer,       sheet_name='Bottom 5 Stores',    index=False)
    yearly.to_excel(writer,        sheet_name='Yearly Summary',     index=False)

# Formatting
HDR_FILL = PatternFill('solid', start_color='0D3B66')
HDR_FONT = Font(bold=True, color='FFFFFF', name='Arial', size=11)
ALT_FILL = PatternFill('solid', start_color='E8F1FB')
SIDE     = Side(style='thin', color='BBBBBB')
BORDER   = Border(left=SIDE, right=SIDE, top=SIDE, bottom=SIDE)

def fmt_sheet(ws):
    for cell in ws[1]:
        cell.font      = HDR_FONT
        cell.fill      = HDR_FILL
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border    = BORDER
    ws.row_dimensions[1].height = 28
    for i, row in enumerate(ws.iter_rows(min_row=2), 2):
        bg = ALT_FILL if i % 2 == 0 else PatternFill()
        for cell in row:
            cell.fill      = bg
            cell.border    = BORDER
            cell.alignment = Alignment(vertical='center')
            if isinstance(cell.value, float): cell.number_format = '#,##0.00'
            elif isinstance(cell.value, int):  cell.number_format = '#,##0'
    for col in ws.columns:
        w = max((len(str(c.value or '')) for c in col), default=10)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(w + 4, 35)
    ws.freeze_panes = 'A2'

wb = load_workbook(output)
for name in wb.sheetnames:
    fmt_sheet(wb[name])
wb.save(output)

print(f'✅ Excel saved → {output}')
print(f'   Sheets: {wb.sheetnames}')

In [ ]:
# ── Cell 12: Final KPI Summary ────────────────────────────────────────────────
print('=' * 55)
print('  WALMART DATASET — ANALYSIS COMPLETE')
print('=' * 55)
print(f'  Rows          : {len(df):,}')
print(f'  Stores        : {df["Store"].nunique()}')
print(f'  Date Range    : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'  Total Revenue : ${df["Weekly_Sales"].sum():,.2f}')
print(f'  Avg Wkly Sales: ${df["Weekly_Sales"].mean():,.2f}')
print(f'  Holiday Weeks : {(df["Holiday_Flag"]=="Holiday").sum()}')
print('-' * 55)
print('  Top 3 Stores:')
for _, row in store_summary.head(3).iterrows():
    print(f'    Store {int(row["Store"]):>2}  →  ${row["Total_Sales"]:>14,.2f}')
print('=' * 55)
print(f'  📁 Excel  → walmart_analysis_output.xlsx')
print(f'  📊 Charts → walmart_charts.png')